In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import re

# Veri setini okuma (hocanın yönergedeki encoding formatıyla)
df = pd.read_csv('/content/drive/MyDrive/Buyuk_Veri_Donem_Projesi/twitter_big_data_pipeline/data/raw/Tweets.csv', encoding='latin-1')

# Veri setinin keşfi: boyut, sütun isimleri ve örnek kayıtlar
print("Toplam Satır ve Sütun Sayısı:", df.shape)
print("\nSütun İsimleri:\n", df.columns.tolist())

# İlk 5 satırı görelim
df.head()

Toplam Satır ve Sütun Sayısı: (14640, 15)

Sütun İsimleri:
 ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [ ]:
print("--- Veri Tipleri ve Eksik (Null) Değer Durumu ---")
df.info()

print("\n--- Temel İstatistikler ---")
df.describe(include='all')

--- Veri Tipleri ve Eksik (Null) Değer Durumu ---
<class 'pandas.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  str    
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   str    
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  str    
 6   airline_sentiment_gold        40 non-null     str    
 7   name                          14640 non-null  str    
 8   negativereason_gold           32 non-null     str    
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  str    
 11  tweet_coord                   1019 non-null   str    
 12  tweet_created        

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
count,1.464000e+04,14640,14640.000000,9178,10522.000000,14640,40,14640,32,14640.000000,14640,1019,14640,9907,9820
unique,NaN,3,NaN,10,NaN,6,3,7701,13,NaN,14427,832,14247,3081,85
top,NaN,negative,NaN,Customer Service Issue,NaN,United,negative,JetBlueNews,Customer Service Issue,NaN,@united thanks,"[0.0, 0.0]",2015-02-24 09:54:34 -0800,"Boston, MA",Eastern Time (US & Canada)
freq,NaN,9178,NaN,2910,NaN,3822,32,63,12,NaN,6,164,5,157,3744
mean,5.692184e+17,NaN,0.900169,NaN,0.638298,NaN,NaN,NaN,NaN,0.082650,NaN,NaN,NaN,NaN,NaN
std,7.791112e+14,NaN,0.162830,NaN,0.330440,NaN,NaN,NaN,NaN,0.745778,NaN,NaN,NaN,NaN,NaN
min,5.675883e+17,NaN,0.335000,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
25%,5.685592e+17,NaN,0.692300,NaN,0.360600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
50%,5.694779e+17,NaN,1.000000,NaN,0.670600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
75%,5.698905e+17,NaN,1.000000,NaN,1.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Temizleme öncesi kayıt sayısını tutalım
baslangic_kayit = len(df)
print(f"Temizleme öncesi toplam kayıt: {baslangic_kayit}")

def metin_temizle(text):
    if type(text) != str:
        return text

    # 1. URL'leri temizleme
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # 2. Mention (@kullanici) temizleme
    text = re.sub(r'@\w+', '', text)
    # 3. Hashtag (#) işaretini silme (kelime kalıyor)
    text = re.sub(r'#', '', text)
    # 4. Özel karakterleri temizleme (Sadece harf, rakam ve boşluklar kalsın)
    text = re.sub(r'[^\w\s]', '', text)

    # Fazla boşlukları alıp küçük harfe çevirelim
    text = " ".join(text.split()).lower()
    return text

# Fonksiyonu uygulayıp yeni sütun oluşturalım
df['cleaned_text'] = df['text'].apply(metin_temizle)

# Temizlenmiş haliyle eski halini karşılaştıralım
df[['text', 'cleaned_text']].head()


Temizleme öncesi toplam kayıt: 14640


,text,cleaned_text
0,@VirginAmerica What @dhepburn said.,what said
1,@VirginAmerica plus you've added commercials t...,plus youve added commercials to the experience...
2,@VirginAmerica I didn't today... Must mean I n...,i didnt today must mean i need to take another...
3,@VirginAmerica it's really aggressive to blast...,its really aggressive to blast obnoxious enter...
4,@VirginAmerica and it's a really big bad thing...,and its a really big bad thing about it


In [ ]:
# Tekrarlanan (duplicate) kayıtları silme
df = df.drop_duplicates()

# Eksik (null) değerlerin tespiti ve giderilmesi
# En önemli boşluk 'negativereason' sütununda. Orayı dolduruyoruz.
df['negativereason'] = df['negativereason'].fillna('Belirtilmedi')

# Kullanılmayacak ama null olan diğer sütunları da standart bir değerle dolduralım
df['airline_sentiment_gold'] = df['airline_sentiment_gold'].fillna('Bilinmiyor')
df['negativereason_gold'] = df['negativereason_gold'].fillna('Bilinmiyor')
df['tweet_coord'] = df['tweet_coord'].fillna('Bilinmiyor')

# Sütun veri tiplerinin doğrulanması (tarihi datetime yapıyoruz)
df['tweet_created'] = pd.to_datetime(df['tweet_created'])

# Temizleme sonrası durumu raporlama
bitis_kayit = len(df)
print(f"\nTemizleme sonrası toplam kayıt: {bitis_kayit}")
print(f"Silinen tekrarlı (duplicate) kayıt sayısı: {baslangic_kayit - bitis_kayit}")


Temizleme sonrası toplam kayıt: 14604
Silinen tekrarlı (duplicate) kayıt sayısı: 36


In [ ]:
print("--- VERİ KALİTESİ DEĞERLENDİRMESİ ---")

# 1. Tamlık (Completeness)
print("\n1. Doğruluk / Tamlık (Completeness):")
print("Veri setindeki eksik (null) değerler 'Belirtilmedi' veya 'Bilinmiyor' olarak dolduruldu. Veri kaybı olmadan tamlık sağlandı.")

# 2. Tutarlılık (Consistency)
print("\n2. Tutarlılık (Consistency):")
print(f"Veri setindeki {baslangic_kayit - bitis_kayit} adet tekrar eden kayıt silinerek verilerin tekil olması sağlandı.")

# 3. Geçerlilik (Validity)
print("\n3. Geçerlilik (Validity):")
print("Tarih verisi tutan 'tweet_created' sütunu metin tipindeydi, geçerli bir datetime formatına çevrildi. Metinler gereksiz karakterlerden arındırıldı.")

--- VERİ KALİTESİ DEĞERLENDİRMESİ ---

1. Doğruluk / Tamlık (Completeness):
Veri setindeki eksik (null) değerler 'Belirtilmedi' veya 'Bilinmiyor' olarak dolduruldu. Veri kaybı olmadan tamlık sağlandı.

2. Tutarlılık (Consistency):
Veri setindeki 36 adet tekrar eden kayıt silinerek verilerin tekil olması sağlandı.

3. Geçerlilik (Validity):
Tarih verisi tutan 'tweet_created' sütunu metin tipindeydi, geçerli bir datetime formatına çevrildi. Metinler gereksiz karakterlerden arındırıldı.


In [ ]:
# Önceki dosyada temizlediğimiz veriyi processed klasörüne kaydediyoruz
df.to_csv('/content/drive/MyDrive/Buyuk_Veri_Donem_Projesi/twitter_big_data_pipeline/data/processed/cleaned_tweets.csv', index=False)